# Paper 01: Data Preparation
Load weekly climate data (MERRA-2) and cattle slaughter data (USDA), 
engineer features, and export analysis-ready CSV for modeling.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path('/Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load .env from project root
load_dotenv(project_root / '.env')

import config
from src.analysis import load_weekly_data
from src.data_loading import load_region_mask, create_cattle_region_masks, load_cattle_data
from src.aggregation import aggregate_to_region
from src.weekly_operations import (
    reshape_cattle_to_long, merge_cattle_climate, 
    create_lagged_variables, add_cyclical_time_features,
    create_zero_inflated_features
)

print(f"Project root: {project_root}")
print(f"Data directory: {config.CATTLE_DATA_DIR}")
print(f"NASS_API_KEY: {'set' if os.getenv('NASS_API_KEY') else 'not set'}")

Project root: /Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research
Data directory: /Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/data/cattle_data
NASS_API_KEY: set


## 1. Load Weekly Climate Data
Load pre-processed weekly NetCDF files with automatic timedelta→hours conversion.

In [2]:
ds_night, ds_day, ds_vpd, week_dates = load_weekly_data(
    config.PROCESSED_WEEKLY_DIR, 
    config.CATTLE_DATA_FILE
)

print(f"\nNighttime variables: {list(ds_night.data_vars)}")
print(f"Daytime variables: {list(ds_day.data_vars)}")
print(f"VPD variables: {list(ds_vpd.data_vars)}")
print(f"\nSample nighttime values (should be 0-70 range, NOT 1e11):")
print(f"  hours_above_21 max: {float(ds_night['hours_above_21'].max()):.1f}")
print(f"  hours_above_24 max: {float(ds_night['hours_above_24'].max()):.1f}")

/Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/src/analysis.py:64: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds_night = xr.open_dataset(processed_weekly_dir / 'weekly_nighttime_recovery.nc')
/Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/src/analysis.py:65: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds_day = xr.open_dataset(processed_weekly_dir / 'weekly_daytime_heat.nc')


Climate data coverage: 2191 weeks
  From: 1984-01-07
  To: 2025-12-27

Cattle data: 2254 total weeks
  Overlapping with climate data: 2191 weeks

Nighttime variables: ['hours_below_18_above_0', 'hours_below_21_above_0', 'hours_below_24_above_0', 'hours_above_21', 'hours_above_24', 'hours_below_0', 'hours_below_neg5', 'hours_below_neg10']
Daytime variables: ['hours_above_25', 'hours_above_30', 'hours_above_35', 'hours_above_40', 'hours_below_neg5', 'hours_below_0', 'hours_below_5']
VPD variables: ['vpd_mean', 'vpd_min', 'vpd_max']

Sample nighttime values (should be 0-70 range, NOT 1e11):
  hours_above_21 max: 70.0
  hours_above_24 max: 70.0


## 2. Create Region Masks and Compute Regional Means
Aggregate gridded data to USDA cattle regions 4 (Southeast) and 6 (South Central).

In [3]:
# Load mask and create region masks
mask_ds, _ = load_region_mask(config.MASK_FILE)
region_masks = create_cattle_region_masks(mask_ds, config.CATTLE_REGION_IDS)

print("Region masks created:")
for name, mask in region_masks.items():
    print(f"  {name}: {int(mask.sum())} grid cells")

# Cache file for regional climate means (avoids ~3 min recompute)
cache_file = config.PROCESSED_WEEKLY_DIR / 'climate_weekly_regional.csv'

if cache_file.exists():
    climate_weekly = pd.read_csv(cache_file, parse_dates=['week_ending'])
    print(f"\nLoaded cached climate data: {climate_weekly.shape}")
    print(f"Columns: {list(climate_weekly.columns)}")
else:
    print("\nNo cache found — computing regional means (vectorized)...")
    
    # Vectorized aggregation: apply mask once, mean over all weeks at once
    regional_frames = []
    
    for region_name, region_mask in region_masks.items():
        region_records = {}
        region_records['week_ending'] = pd.to_datetime(week_dates)
        region_records['region'] = region_name
        
        # Nighttime variables — vectorized over all weeks
        for var in ds_night.data_vars:
            vals = ds_night[var].where(region_mask).mean(dim=['lat', 'lon']).values
            region_records[f'mean_nighttime_{var}'] = vals.astype(float)
        
        # Daytime variables
        for var in ds_day.data_vars:
            vals = ds_day[var].where(region_mask).mean(dim=['lat', 'lon']).values
            region_records[f'mean_daytime_{var}'] = vals.astype(float)
        
        # VPD variables
        for var in ds_vpd.data_vars:
            vals = ds_vpd[var].where(region_mask).mean(dim=['lat', 'lon']).values
            region_records[f'mean_{var}'] = vals.astype(float)
        
        regional_frames.append(pd.DataFrame(region_records))
        print(f"  {region_name}: {len(week_dates)} weeks computed")
    
    climate_weekly = pd.concat(regional_frames, ignore_index=True)
    climate_weekly = climate_weekly.sort_values(['region', 'week_ending']).reset_index(drop=True)
    
    # Cache to CSV
    climate_weekly.to_csv(cache_file, index=False)
    print(f"\nCached to: {cache_file}")
    print(f"Combined climate data: {climate_weekly.shape}")
    print(f"Columns: {list(climate_weekly.columns)}")

Region masks created:
  region_4: 313 grid cells
  region_6: 437 grid cells

Loaded cached climate data: (4382, 20)
Columns: ['week_ending', 'region', 'mean_nighttime_hours_below_18_above_0', 'mean_nighttime_hours_below_21_above_0', 'mean_nighttime_hours_below_24_above_0', 'mean_nighttime_hours_above_21', 'mean_nighttime_hours_above_24', 'mean_nighttime_hours_below_0', 'mean_nighttime_hours_below_neg5', 'mean_nighttime_hours_below_neg10', 'mean_daytime_hours_above_25', 'mean_daytime_hours_above_30', 'mean_daytime_hours_above_35', 'mean_daytime_hours_above_40', 'mean_daytime_hours_below_neg5', 'mean_daytime_hours_below_0', 'mean_daytime_hours_below_5', 'mean_vpd_mean', 'mean_vpd_min', 'mean_vpd_max']


## 2b. Add Soil Moisture and Precipitation

Load weekly RZMC (root zone soil moisture) and PRECIP_TOTAL from the pre-processed
`weekly_soil_precip.nc`. Compute regional means and soil moisture anomalies
(deviation from weekly climatology) to identify drought stress independent of
normal seasonal drying.

In [4]:
# Load weekly soil moisture and precipitation
soil_precip_file = config.PROCESSED_WEEKLY_DIR / 'weekly_soil_precip.nc'

if soil_precip_file.exists():
    ds_soil = xr.open_dataset(soil_precip_file)
    soil_weeks = ds_soil['week'].values
    print(f"Soil/precip data: {len(soil_weeks)} weeks")
    print(f"  RZMC range: [{float(ds_soil['RZMC'].min()):.4f}, {float(ds_soil['RZMC'].max()):.4f}] kg/m²")
    print(f"  PRECIP range: [{float(ds_soil['PRECIP_TOTAL'].min()):.2f}, {float(ds_soil['PRECIP_TOTAL'].max()):.2f}] mm/week")
    
    # Compute regional means for RZMC and PRECIP_TOTAL
    soil_regional_frames = []
    
    for region_name, region_mask in region_masks.items():
        rzmc_regional = ds_soil['RZMC'].where(region_mask).mean(dim=['lat', 'lon']).values
        precip_regional = ds_soil['PRECIP_TOTAL'].where(region_mask).mean(dim=['lat', 'lon']).values
        
        soil_df = pd.DataFrame({
            'week_ending': pd.to_datetime(soil_weeks),
            'region': region_name,
            'mean_rzmc': rzmc_regional.astype(float),
            'mean_precip_total': precip_regional.astype(float),
        })
        soil_regional_frames.append(soil_df)
        print(f"  {region_name}: RZMC mean={soil_df['mean_rzmc'].mean():.4f}, PRECIP mean={soil_df['mean_precip_total'].mean():.2f} mm/wk")
    
    soil_weekly = pd.concat(soil_regional_frames, ignore_index=True)
    
    # Compute RZMC anomaly per region (deviation from weekly climatology)
    # Climatology computed from training period only (1984-2015)
    soil_weekly['week_of_year'] = pd.to_datetime(soil_weekly['week_ending']).dt.isocalendar().week.astype(int)
    soil_weekly['year'] = pd.to_datetime(soil_weekly['week_ending']).dt.year
    
    train_soil = soil_weekly[soil_weekly['year'] <= config.MODEL_TRAIN_END]
    
    for region_name in soil_weekly['region'].unique():
        # Climatology: mean RZMC for each week-of-year from training period
        r_train = train_soil[train_soil['region'] == region_name]
        climatology = r_train.groupby('week_of_year')['mean_rzmc'].mean()
        
        # Anomaly = observed - climatology
        mask = soil_weekly['region'] == region_name
        soil_weekly.loc[mask, 'mean_rzmc_anomaly'] = (
            soil_weekly.loc[mask, 'mean_rzmc'].values - 
            soil_weekly.loc[mask, 'week_of_year'].map(climatology).values
        )
    
    # Drought indicator: RZMC anomaly below 20th percentile of training data
    train_anomalies = soil_weekly[soil_weekly['year'] <= config.MODEL_TRAIN_END]['mean_rzmc_anomaly'].dropna()
    drought_threshold = train_anomalies.quantile(0.20)
    soil_weekly['drought_indicator'] = (soil_weekly['mean_rzmc_anomaly'] < drought_threshold).astype(int)
    
    # Precipitation indicator: above-median weekly precip
    train_precip = soil_weekly[soil_weekly['year'] <= config.MODEL_TRAIN_END]['mean_precip_total']
    precip_threshold = train_precip.median()
    soil_weekly['precip_indicator'] = (soil_weekly['mean_precip_total'] > precip_threshold).astype(int)
    
    # Drop helper columns before merge
    soil_weekly = soil_weekly.drop(columns=['week_of_year', 'year'])
    
    print(f"\nSoil/precip features: {list(soil_weekly.columns)}")
    print(f"  Drought threshold (20th pctl anomaly): {drought_threshold:.4f} kg/m²")
    print(f"  Precip threshold (median): {precip_threshold:.2f} mm/week")
    print(f"  Drought weeks: {soil_weekly['drought_indicator'].sum()} / {len(soil_weekly)} ({100*soil_weekly['drought_indicator'].mean():.1f}%)")
    
    # Merge into climate_weekly
    climate_weekly = climate_weekly.merge(soil_weekly, on=['week_ending', 'region'], how='left')
    print(f"\nClimate data with soil/precip: {climate_weekly.shape}")
    
    ds_soil.close()
else:
    print(f"WARNING: {soil_precip_file} not found")
    print("Run scripts/process_weekly_soil_precip.py first to generate weekly data")

Soil/precip data: 2192 weeks
  RZMC range: [0.0536, 0.4502] kg/m²
  PRECIP range: [0.00, 3848.69] mm/week
  region_4: RZMC mean=0.2858, PRECIP mean=25.13 mm/wk
  region_6: RZMC mean=0.2351, PRECIP mean=15.21 mm/wk

Soil/precip features: ['week_ending', 'region', 'mean_rzmc', 'mean_precip_total', 'mean_rzmc_anomaly', 'drought_indicator', 'precip_indicator']
  Drought threshold (20th pctl anomaly): -0.0187 kg/m²
  Precip threshold (median): 17.21 mm/week
  Drought weeks: 894 / 4384 (20.4%)

Climate data with soil/precip: (4382, 25)


## 3. Load and Merge Cattle Slaughter Data

In [5]:
# Load cattle data
cattle_df = load_cattle_data(config.CATTLE_DATA_FILE)
print(f"Cattle data: {len(cattle_df)} weeks, columns: {list(cattle_df.columns)[:8]}...")

# Reshape to long format (one row per region per week)
cattle_long = reshape_cattle_to_long(cattle_df)
print(f"\nCattle long format: {cattle_long.shape}")
print(cattle_long.groupby('region')[['slaughter_beef_dairy', 'slaughter_dairy']].describe().round(1))

# Merge climate + cattle
merged_df = merge_cattle_climate(climate_weekly, cattle_long)
print(f"\nMerged dataset: {merged_df.shape}")

Cattle data: 2254 weeks, columns: ['date', 'week', 'region_1_2_dairy', 'region_1_2_beef_dairy', 'region_3_dairy', 'region_3_beef_dairy', 'region_4_dairy', 'region_4_beef_dairy']...

Cattle long format: (4508, 4)
         slaughter_beef_dairy                                          \
                        count  mean  std  min   25%   50%   75%   max   
region                                                                  
region_4               2193.0  13.3  3.4  5.6  11.0  13.0  14.7  29.8   
region_6               2193.0  19.1  4.8  4.6  15.4  19.0  22.0  40.6   

         slaughter_dairy                                      
                   count mean  std  min  25%  50%  75%   max  
region                                                        
region_4          2141.0  4.1  1.8  1.6  2.9  3.6  5.1  14.9  
region_6          2193.0  4.2  2.1  0.7  2.5  3.2  6.1  11.9  
Merged data shape: (4382, 28)

Merge statistics:
_merge
both          4382
left_only        0
right_only   

## 3b. USDA NASS Cattle Inventory Data (Herd Size Normalization)
Pull annual cattle inventory (total head) by state from USDA NASS QuickStats API.
This allows us to normalize slaughter counts by herd size — confirming that changes
in slaughter are driven by heat stress, not simply by more cattle in a region.

In [6]:
import requests
import json

# USDA NASS QuickStats API
# API key loaded from .env via dotenv (set NASS_API_KEY in your .env file)
NASS_API_KEY = os.getenv('NASS_API_KEY', 'YOUR_API_KEY_HERE')

NASS_BASE_URL = 'https://quickstats.nass.usda.gov/api/api_GET/'

def fetch_nass_cattle_inventory(api_key, years_range=(1984, 2025)):
    """Fetch annual cattle inventory (total head) from USDA NASS QuickStats.
    
    Uses 'CATTLE, INCL CALVES - INVENTORY' at FIRST OF JAN each year
    as the annual herd size estimate per state.
    """
    all_data = []
    
    region_states = {
        'region_4': ['AL', 'FL', 'GA', 'KY', 'MS', 'NC', 'SC', 'TN'],
        'region_6': ['AR', 'LA', 'NM', 'OK', 'TX']
    }
    
    for region, state_list in region_states.items():
        for state_abbr in state_list:
            params = {
                'key': api_key,
                'commodity_desc': 'CATTLE',
                'statisticcat_desc': 'INVENTORY',
                'unit_desc': 'HEAD',
                'domain_desc': 'TOTAL',
                'agg_level_desc': 'STATE',
                'state_alpha': state_abbr,
                'year__GE': str(years_range[0]),
                'year__LE': str(years_range[1]),
                'freq_desc': 'POINT IN TIME',
                'reference_period_desc': 'FIRST OF JAN',
                'short_desc': 'CATTLE, INCL CALVES - INVENTORY',
                'format': 'JSON',
            }
            
            try:
                resp = requests.get(NASS_BASE_URL, params=params, timeout=30)
                if resp.status_code == 200:
                    data = resp.json().get('data', [])
                    for record in data:
                        val = record.get('Value', '').replace(',', '')
                        if val and val != '(D)':
                            all_data.append({
                                'year': int(record['year']),
                                'state': state_abbr,
                                'region': region,
                                'cattle_inventory_head': int(val),
                            })
                    if not data:
                        print(f"  No data for {state_abbr}")
                else:
                    print(f"  API error for {state_abbr}: {resp.status_code}")
            except Exception as e:
                print(f"  Request failed for {state_abbr}: {e}")
    
    if all_data:
        df = pd.DataFrame(all_data)
        print(f"  Fetched {len(df)} state-year records across {df['state'].nunique()} states")
        # Aggregate to regional totals by year
        regional = df.groupby(['year', 'region'])['cattle_inventory_head'].sum().reset_index()
        regional = regional.rename(columns={'cattle_inventory_head': 'regional_inventory_head'})
        return df, regional
    return pd.DataFrame(), pd.DataFrame()

# Fetch data (skip if no API key configured)
if NASS_API_KEY != 'YOUR_API_KEY_HERE':
    print("Fetching cattle inventory from USDA NASS API...")
    state_inventory, regional_inventory = fetch_nass_cattle_inventory(NASS_API_KEY)
    
    if len(regional_inventory) > 0:
        print(f"\nRegional inventory data: {len(regional_inventory)} region-years")
        print(regional_inventory.groupby('region')['regional_inventory_head'].describe().round(0))
        
        # Save for later use
        inventory_file = config.CATTLE_DATA_DIR / 'nass_cattle_inventory.csv'
        regional_inventory.to_csv(inventory_file, index=False)
        state_inventory.to_csv(config.CATTLE_DATA_DIR / 'nass_state_inventory.csv', index=False)
        print(f"\nSaved to: {inventory_file}")
    else:
        print("No data returned from API")
else:
    # Try to load previously saved inventory data
    inventory_file = config.CATTLE_DATA_DIR / 'nass_cattle_inventory.csv'
    if inventory_file.exists():
        regional_inventory = pd.read_csv(inventory_file)
        print(f"Loaded cached inventory data: {len(regional_inventory)} region-years")
    else:
        regional_inventory = pd.DataFrame()
        print("No NASS API key set. To fetch cattle inventory data:")
        print("  1. Register at https://quickstats.nass.usda.gov/api")
        print("  2. Add NASS_API_KEY=your_key to research/.env")
        print("\nThis data is optional — used to normalize slaughter by herd size.")

Fetching cattle inventory from USDA NASS API...
  Fetched 559 state-year records across 13 states

Regional inventory data: 86 region-years
          count        mean        std         min         25%         50%  \
region                                                                       
region_4   43.0  11293837.0  1512246.0   8900000.0  10100000.0  11070000.0   
region_6   43.0  22490581.0  1499034.0  19090000.0  21682500.0  22910000.0   

                 75%         max  
region                            
region_4  12410000.0  14779000.0  
region_6  23430000.0  25010000.0  

Saved to: /Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/data/cattle_data/nass_cattle_inventory.csv


In [7]:
# Merge annual inventory into weekly data for normalization
# This creates slaughter_per_1000_head = (weekly slaughter / annual inventory) * 1000
# so changes in slaughter rate reflect heat stress, not herd size changes

if len(regional_inventory) > 0:
    # Add year to merged_df for joining
    merged_df['year'] = pd.to_datetime(merged_df['week_ending']).dt.year
    
    merged_df = merged_df.merge(
        regional_inventory[['year', 'region', 'regional_inventory_head']],
        on=['year', 'region'],
        how='left'
    )
    
    # Create normalized slaughter rate (per 1000 head of inventory)
    mask = merged_df['regional_inventory_head'].notna() & (merged_df['regional_inventory_head'] > 0)
    merged_df.loc[mask, 'slaughter_rate_per_1000'] = (
        merged_df.loc[mask, 'slaughter_beef_dairy'] / 
        merged_df.loc[mask, 'regional_inventory_head'] * 1e6  # slaughter is in thousands, inventory in head
    )
    
    print("=== Herd-Normalized Slaughter Rate ===")
    print("(Weekly slaughter per 1,000 head of regional inventory)")
    for region in merged_df['region'].unique():
        rdata = merged_df[merged_df['region'] == region]['slaughter_rate_per_1000'].dropna()
        if len(rdata) > 0:
            print(f"\n{region}:")
            print(f"  Mean: {rdata.mean():.2f}, Std: {rdata.std():.2f}")
            print(f"  Inventory coverage: {rdata.count()} / {len(merged_df[merged_df['region'] == region])} weeks")
    
    # Drop year column (will be re-added in feature engineering)
    merged_df = merged_df.drop(columns=['year'], errors='ignore')
else:
    print("Skipping herd normalization — no inventory data available")
    print("Slaughter counts will be used directly (thousands head/week)")

=== Herd-Normalized Slaughter Rate ===
(Weekly slaughter per 1,000 head of regional inventory)

region_4:
  Mean: 1.17, Std: 0.23
  Inventory coverage: 2191 / 2191 weeks

region_6:
  Mean: 0.85, Std: 0.23
  Inventory coverage: 2191 / 2191 weeks


## 4. Feature Engineering
Create lagged variables, rolling windows, cyclical time features, and zero-inflated indicators.

In [8]:
# Create lagged variables (1-8 weeks)
lag_weeks = [1, 2, 3, 4, 8]
merged_lagged = create_lagged_variables(merged_df, lag_weeks=lag_weeks)
print(f"After lags: {merged_lagged.shape[1]} columns (was {merged_df.shape[1]})")

After lags: 182 columns (was 32)


/Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/src/weekly_operations.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lagged_col_name] = df.groupby('region')[col].shift(lag)
/Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/src/weekly_operations.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lagged_col_name] = df.groupby('region')[col].shift(lag)
/Users/klesinger/Library/CloudSt

In [9]:
# Create rolling averages (2, 4, 8 week windows)
# Group by region to avoid cross-region contamination
exclude_cols = ['region', 'week_ending', 'date', '_merge',
                'slaughter_beef_dairy', 'slaughter_dairy',
                'log_slaughter_beef_dairy', 'log_slaughter_dairy']
predictor_cols = [c for c in merged_df.columns if c not in exclude_cols and not c.endswith('_lag')]

for window in config.ROLLING_WINDOWS:
    for col in predictor_cols:
        if col in merged_lagged.columns:
            merged_lagged[f'{col}_roll{window}'] = (
                merged_lagged.groupby('region')[col]
                .transform(lambda x: x.rolling(window, min_periods=1).mean())
            )

print(f"After rolling windows: {merged_lagged.shape[1]} columns")

After rolling windows: 257 columns


/var/folders/jq/m05tkv2d1llbn0w9wc6cltyr0000gn/T/ipykernel_24599/4057215484.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_lagged[f'{col}_roll{window}'] = (
/var/folders/jq/m05tkv2d1llbn0w9wc6cltyr0000gn/T/ipykernel_24599/4057215484.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_lagged[f'{col}_roll{window}'] = (
/var/folders/jq/m05tkv2d1llbn0w9wc6cltyr0000gn/T/ipykernel_24599/4057215484.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 

In [10]:
# Add cyclical time features (sin/cos week-of-year)
merged_lagged = add_cyclical_time_features(merged_lagged, date_col='week_ending')

# Add year and month for temporal analysis
merged_lagged['year'] = pd.to_datetime(merged_lagged['week_ending']).dt.year
merged_lagged['month'] = pd.to_datetime(merged_lagged['week_ending']).dt.month

# Create zero-inflated features for sparse hour counts AND precipitation
zero_inflated_cols = [c for c in merged_lagged.columns 
                      if any(z in c for z in ['hours_above_35', 'hours_above_40', 'hours_above_24', 'precip_total'])
                      and not c.endswith('_has') and not c.endswith('_log1p')
                      and not c.endswith('_indicator')]
merged_lagged = create_zero_inflated_features(merged_lagged, zero_inflated_cols)

# Add interaction features
if 'mean_daytime_hours_above_30' in merged_lagged.columns and 'mean_vpd_mean' in merged_lagged.columns:
    merged_lagged['heat_vpd_interaction'] = (
        merged_lagged['mean_daytime_hours_above_30'] * merged_lagged['mean_vpd_mean']
    )
if 'mean_daytime_hours_above_35' in merged_lagged.columns and 'mean_nighttime_hours_above_21' in merged_lagged.columns:
    merged_lagged['extreme_heat_poor_recovery'] = (
        merged_lagged['mean_daytime_hours_above_35'] * merged_lagged['mean_nighttime_hours_above_21']
    )

# Drought × heat compound stress interaction
if 'mean_rzmc_anomaly' in merged_lagged.columns and 'mean_vpd_mean' in merged_lagged.columns:
    merged_lagged['drought_heat_interaction'] = (
        -merged_lagged['mean_rzmc_anomaly'] * merged_lagged['mean_vpd_mean']
    )  # Negative anomaly (drought) × high VPD = positive compound stress

print(f"Final feature set: {merged_lagged.shape}")
print(f"\nFeature categories:")
base = [c for c in merged_lagged.columns if 'mean_' in c and '_lag' not in c and '_roll' not in c and '_has' not in c and '_log1p' not in c]
lags = [c for c in merged_lagged.columns if '_lag' in c]
rolls = [c for c in merged_lagged.columns if '_roll' in c]
zi = [c for c in merged_lagged.columns if c.endswith('_has') or c.endswith('_log1p')]
indicators = [c for c in merged_lagged.columns if c.endswith('_indicator')]
print(f"  Base climate (incl soil/precip): {len(base)}")
print(f"  Lagged: {len(lags)}")
print(f"  Rolling: {len(rolls)}")
print(f"  Zero-inflated: {len(zi)}")
print(f"  Binary indicators: {len(indicators)}")
print(f"  Interactions: heat_vpd, extreme_heat_poor_recovery, drought_heat")

Final feature set: (4382, 336)

Feature categories:
  Base climate (incl soil/precip): 21
  Lagged: 190
  Rolling: 99
  Zero-inflated: 72
  Binary indicators: 2
  Interactions: heat_vpd, extreme_heat_poor_recovery, drought_heat


## 5. Data Quality Summary

In [11]:
# Drop rows with NaN in outcome (from lags)
analysis_df = merged_lagged.dropna(subset=['slaughter_beef_dairy']).copy()

# Remove merge indicator column
if '_merge' in analysis_df.columns:
    analysis_df = analysis_df.drop(columns=['_merge'])

print("=== Dataset Summary ===")
print(f"Shape: {analysis_df.shape}")
print(f"Regions: {analysis_df['region'].unique()}")
print(f"Date range: {analysis_df['week_ending'].min()} to {analysis_df['week_ending'].max()}")
print(f"Weeks per region: {analysis_df.groupby('region').size().to_dict()}")

print("\n=== Target Variable (Slaughter, thousands head/week) ===")
for region in analysis_df['region'].unique():
    rdata = analysis_df[analysis_df['region'] == region]['slaughter_beef_dairy']
    print(f"\n{region}:")
    print(f"  Mean: {rdata.mean():.1f}, Std: {rdata.std():.1f}")
    print(f"  Min: {rdata.min():.1f}, Max: {rdata.max():.1f}")
    print(f"  Skewness: {rdata.skew():.3f}")

print("\n=== Missing Values ===")
missing = analysis_df.isnull().sum()
missing_pct = (missing / len(analysis_df) * 100)
has_missing = missing[missing > 0]
if len(has_missing) > 0:
    print(f"Columns with missing values: {len(has_missing)}")
    for col in has_missing.index[:10]:
        print(f"  {col}: {has_missing[col]} ({missing_pct[col]:.1f}%)")
else:
    print("No missing values in any column")

# Train/test split summary
train = analysis_df[analysis_df['year'] <= config.MODEL_TRAIN_END]
test = analysis_df[analysis_df['year'] >= config.MODEL_TEST_START]
print(f"\n=== Train/Test Split ===")
print(f"Train (≤{config.MODEL_TRAIN_END}): {len(train)} samples")
print(f"Test (≥{config.MODEL_TEST_START}): {len(test)} samples")

=== Dataset Summary ===
Shape: (4382, 335)
Regions: ['region_4' 'region_6']
Date range: 1984-01-07 00:00:00 to 2025-12-27 00:00:00
Weeks per region: {'region_4': 2191, 'region_6': 2191}

=== Target Variable (Slaughter, thousands head/week) ===

region_4:
  Mean: 13.3, Std: 3.4
  Min: 5.7, Max: 29.8
  Skewness: 1.302

region_6:
  Mean: 19.1, Std: 4.8
  Min: 4.6, Max: 40.6
  Skewness: 0.418

=== Missing Values ===
Columns with missing values: 172
  slaughter_dairy: 52 (1.2%)
  log_slaughter_dairy: 52 (1.2%)
  mean_nighttime_hours_below_18_above_0_lag1: 2 (0.0%)
  mean_nighttime_hours_below_21_above_0_lag1: 2 (0.0%)
  mean_nighttime_hours_below_24_above_0_lag1: 2 (0.0%)
  mean_nighttime_hours_above_21_lag1: 2 (0.0%)
  mean_nighttime_hours_above_24_lag1: 2 (0.0%)
  mean_nighttime_hours_below_0_lag1: 2 (0.0%)
  mean_nighttime_hours_below_neg5_lag1: 2 (0.0%)
  mean_nighttime_hours_below_neg10_lag1: 2 (0.0%)

=== Train/Test Split ===
Train (≤2015): 3338 samples
Test (≥2016): 1044 samples


## 6. Export Analysis-Ready CSV

In [12]:
# Ensure output directory exists
config.PAPER_ANALYSIS_FILE.parent.mkdir(parents=True, exist_ok=True)

# Export
analysis_df.to_csv(config.PAPER_ANALYSIS_FILE, index=False)
print(f"Exported analysis-ready CSV: {config.PAPER_ANALYSIS_FILE}")
print(f"File size: {config.PAPER_ANALYSIS_FILE.stat().st_size / 1024 / 1024:.1f} MB")
print(f"Shape: {analysis_df.shape}")
print(f"Columns: {len(analysis_df.columns)}")

Exported analysis-ready CSV: /Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research/data/cattle_data/paper_analysis_ready.csv
File size: 17.8 MB
Shape: (4382, 335)
Columns: 335


## 7. Column Reference
Quick reference of all columns in the exported dataset.

In [13]:
print("=== All Columns ===\n")
for i, col in enumerate(analysis_df.columns):
    dtype = analysis_df[col].dtype
    sample = analysis_df[col].iloc[0] if len(analysis_df) > 0 else 'N/A'
    print(f"{i+1:3d}. {col:<55s} {str(dtype):<12s} sample: {sample}")

=== All Columns ===

  1. week_ending                                             datetime64[ns] sample: 1984-01-07 00:00:00
  2. region                                                  object       sample: region_4
  3. mean_nighttime_hours_below_18_above_0                   float64      sample: 48.894568690095845
  4. mean_nighttime_hours_below_21_above_0                   float64      sample: 49.1629392971246
  5. mean_nighttime_hours_below_24_above_0                   float64      sample: 49.1629392971246
  6. mean_nighttime_hours_above_21                           float64      sample: 0.0
  7. mean_nighttime_hours_above_24                           float64      sample: 0.0
  8. mean_nighttime_hours_below_0                            float64      sample: 20.8370607028754
  9. mean_nighttime_hours_below_neg5                         float64      sample: 2.779552715654952
 10. mean_nighttime_hours_below_neg10                        float64      sample: 0.4249201277955272
 11. mean_day